---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png">

<h1 align="center">Department of Data Science</h1>
<h1 align="center">Course: Generative and Agentic AI</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-25: Fine-Tuning TinyLlama using <b>TRL</b></h1>

# Learning Agenda of This Notebook

1. Check your Runtime and Install Dependencies (installation needed only on Google Colab)
2. Dataset Curation
3. Configuration of Model, Dataset, LoRA Adapters, and Training Hyperparameters
4. Load Base Model + Tokenizer
5. Prepare Dataset
6. Configure and Inject LoRA Adapters
7. Configure Training Parameters
8. Train the Model (needs an NVIDIA GPU: your own machine or Google Colab)
9. Training Loss Plot
10. Save the Adapter Weights, Push to HF and Do Inference (`arif-butt/tinyllama-trl-lora-adapter`)
11. Merge Adapter Weights, Push to HF and Do Inference (`arif-butt/tinyllama-trl-merged`)
12. Generate GGUF and Push to HF and Do Inference via Ollama and Llama.cpp (`arif-butt/tinyllama-trl-gguf`)

# <span style='background :lightgreen' >1. Check your Runtime and Install Dependencies</span>

This notebook runs **both on your own computer and on Google Colab**. No NVIDIA GPU on your computer? Click this button to open this notebook in Google Colab:

<a href="https://colab.research.google.com/github/arifpucit/Generative-and-Agentic-AI/blob/main/Notebooks/Lec_25_%28FineTuning_TRL%29.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


| Where | What to do |
|---|---|
| **Your computer** | Create the environment once with `uv sync` (from the repository root) and open the notebook with the project kernel. **Skip cell 1b** — the virtual environment already has every package. |
| **Google Colab** | Click the **Open in Colab** button above (or the link printed by cell 1a). In Colab select `Runtime > Change runtime type > T4 GPU`, then run cell **1a** and cell **1b** (1b installs the packages), and then the rest of the notebook. |

- Every cell runs on CPU, Mac or GPU **except the training cell**, which needs an **NVIDIA (CUDA) GPU with at least 8 GB** of memory (training peaks at about 5–7 GB, depending on the speed settings chosen for your GPU in section 3).
- On a machine without such a GPU, the training cell prints a message and is skipped. The later cells then use the fine-tuned model that is already published on the Hugging Face Hub, so you can still follow the rest of the notebook.

## a. Check your runtime (always run this cell first)

In [ ]:
# ── Run this cell FIRST ─────────────────────────────────────────────────────────────────────
# It tells you whether the TRAINING cell of this notebook can run on this machine or whether you need Google Colab.
# All other cells run everywhere (CPU, Mac, Colab); only the training cell needs an NVIDIA (CUDA) GPU.
import sys, platform, shutil, pathlib

IN_COLAB    = "google.colab" in sys.modules   # True when this notebook runs inside Google Colab
MIN_VRAM_GB = 8                             # GPU memory (GB) the training cell of this notebook needs
CAN_TRAIN   = False                           # becomes True below if a suitable CUDA GPU is found
TRAINED     = False                           # becomes True after the training cell finishes successfully
COLAB_URL   = "https://colab.research.google.com/github/arifpucit/Generative-and-Agentic-AI/blob/main/Notebooks/Lec_25_%28FineTuning_TRL%29.ipynb"   # this notebook in Google Colab

# Windows fix: `trl` reads its bundled chat-template files with Windows' default encoding (cp1252) and crashes on
# import with a UnicodeDecodeError. Make pathlib read text files as UTF-8 by default for this session.
if sys.platform == "win32" and not sys.flags.utf8_mode:
    _read_text = pathlib.Path.read_text
    pathlib.Path.read_text = lambda self, encoding=None, *args, **kwargs: _read_text(self, encoding or "utf-8", *args, **kwargs)

import torch

print(f"Runtime : {'Google Colab' if IN_COLAB else 'Local machine'} ({platform.system()}, Python {platform.python_version()})")
print(f"PyTorch : {torch.__version__} (CUDA build: {torch.version.cuda or 'none, this is a CPU-only build'})")

if torch.cuda.is_available() and torch.cuda.device_count() > 0:
    major, minor = torch.cuda.get_device_capability(0)
    total_gb     = torch.cuda.get_device_properties(0).total_memory / 1e9
    free_gb      = torch.cuda.mem_get_info()[0] / 1e9
    print(f"GPU     : {torch.cuda.get_device_name(0)} (compute capability {major}.{minor})")
    print(f"VRAM    : {total_gb:.1f} GB total, {free_gb:.1f} GB free right now")
    print(f"bf16    : {'supported' if torch.cuda.is_bf16_supported() else 'not supported (fp16 is used instead)'}")
    if f"sm_{major}{minor}" not in torch.cuda.get_arch_list():
        print(f"\n⚠️  This PyTorch build has no kernels for your GPU (sm_{major}{minor}); supported: {', '.join(torch.cuda.get_arch_list())}")
    elif total_gb < MIN_VRAM_GB:
        print(f"\n⚠️  Training in this notebook needs about {MIN_VRAM_GB} GB of GPU memory, your GPU has {total_gb:.1f} GB.")
    else:
        CAN_TRAIN = True
        if free_gb < MIN_VRAM_GB:
            print(f"\n⚠️  Only {free_gb:.1f} GB is free right now: close other programs that use the GPU, or training may run out of memory.")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("GPU     : Apple Silicon (MPS) - fine for inference, but training in this notebook needs an NVIDIA CUDA GPU")
elif shutil.which("nvidia-smi") and torch.version.cuda is None:
    print("GPU     : an NVIDIA GPU is present, but the installed PyTorch is a CPU-only build!")
    print("          Fix: run `uv sync` again from the repository root (it installs the CUDA build), then restart the kernel.")
else:
    print("GPU     : none detected")

print("\n" + "=" * 100)
if CAN_TRAIN:
    print("✅ You CAN run the training cell of this notebook on this machine.")
elif IN_COLAB:
    print("❌ This Colab session has no usable GPU: select Runtime > Change runtime type > T4 GPU, then run this cell again.")
else:
    print("❌ You can NOT run the training cell on this machine: open this notebook in Google Colab (T4 GPU) to train:")
    print(f"   {COLAB_URL}")
    print("   In Colab: Runtime > Change runtime type > T4 GPU, then run cell 1a, cell 1b and the rest of the notebook.")
    print("   All other cells still run here; the training cell and the cells that need its output print a message and skip.")
print("=" * 100)

## b. Install dependencies (Google Colab only)
- On your own computer this cell does nothing, because `uv sync` has already installed everything into the virtual environment.
- On Colab it installs the same versions that the course's `uv.lock` uses, so both places run identical code.

```
`transformers` | Load pretrained models and tokenizers from HuggingFace
`peft`         | Provides `LoraConfig` and `get_peft_model()` for LoRA adapter injection
`datasets`     | Load datasets from HuggingFace Hub
`accelerate`   | Enables efficient GPU training
`trl`          | Training utilities (`SFTTrainer`, `SFTConfig`)
`python-dotenv`| Loads API keys / tokens from a `.env` file
```

In [ ]:
# ── Run this cell ONLY on Google Colab (it skips itself on your own computer) ────────────────────
import sys, subprocess

if "google.colab" in sys.modules:
    import importlib.metadata, torch
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers==5.7.0",
        "peft==0.19.1",
        "torchao==0.18.0",      # Colab ships torchao 0.10, which makes peft fail with "incompatible version of torchao"
        "trl==1.3.0",
        "accelerate==1.13.0",
        "datasets==4.8.5",
        "python-dotenv",
    ], check=True)
    if importlib.metadata.version("torch").split("+")[0] != torch.__version__.split("+")[0]:
        print("⚠️  pip upgraded PyTorch: select Runtime > Restart session, then run the notebook again from the top.")
    print("✅ All packages installed successfully!")
else:
    print("Local machine: nothing to install, the uv virtual environment already has every package.")

# <span style='background :lightgreen' >2. Dataset Curation</span>
- The dataset used for fine-tuning is a **custom synthetic Q&A dataset** built around **Dr. Muhammad Arif Butt** — covering his academic background, teaching experience, research interests, contact information and YouTube channel — curated specifically to teach the model to answer questions about him accurately and naturally.
## How the Dataset Was Curated
The dataset was created following a six-step curation pipeline — the same pipeline students can follow to build their own domain-specific datasets:
- **Step 1 — Collection:** Raw facts were gathered from Dr. Arif Butt's official website (https://arifbutt.me) and YouTube channel — covering all relevant topics including academic position, qualifications, research interests, courses taught and contact details
- **Step 2 — AI-Assisted Q&A Generation:** Each collected fact was fed into an AI model with a prompt instructing it to generate **3 to 5 different question variations per fact** — this rephrasing improves model generalisation by teaching it to answer the same fact regardless of how the question is phrased:
```
Fact: "Dr. Arif Butt is an Assistant Professor at PUCIT"

Generated question variations:
  "Who is Dr. Muhammad Arif Butt?"
  "Tell me about Dr. Arif Butt"
  "What is Arif Butt's current position?"
  "Where does Dr. Muhammad Arif Butt work?"
  "Which university does Dr. Arif Butt teach at?"
```
- **Step 3 — Cleaning and Validation:** All generated Q&A pairs were reviewed to remove duplicates, fix factual inaccuracies, ensure consistent tone and style, and discard any incomplete or low quality examples — human review of every example is critical at this stage since AI-generated data can contain subtle errors
- **Step 4 — Formatting:** Each Q&A pair was structured into the standard **chat `messages` format** (a list of `{role, content}` dicts). Later in this notebook every example is converted into TinyLlama-Chat's prompt template — this is not optional — the chat model was fine-tuned on that exact template and responds best when the fine-tuning data matches it:
```json
{
  "messages": [
    {"role": "user",      "content": "Who is Dr. Muhammad Arif Butt?"},
    {"role": "assistant", "content": "Dr. Muhammad Arif Butt is an Assistant Professor
                                      at the Department of Data Science,
                                      University of the Punjab..."}
  ]
}
```
- **Step 5 — Balancing:** Questions were distributed evenly across all topics — academic background, teaching experience, research interests, contact information and YouTube channel — preventing the model from becoming biased toward one topic while forgetting others
- **Step 6 — Upload to HuggingFace Hub:** The final cleaned and formatted dataset was saved as `.jsonl` and uploaded as a public dataset so it can be loaded in one line anywhere:

#### 🔗 **Dataset:** [arif-butt/arifbutt_dataset](https://huggingface.co/datasets/arif-butt/arifbutt_dataset)


## Students should build their own synthetic Dataset
- Pick any domain, a product, a company or a public figure.
- Collect facts from public sources;  websites, Wikipedia, official pages.
- Feed the facts into an AI model and ask it to generate 3 to 5 question variations per fact.
- Review and clean all generated pairs, remove duplicates and fix any inaccuracies.
- Format each pair as chat `messages` (role/content) and save as `.jsonl`.
- Upload to HuggingFace Hub using `dataset.push_to_hub()` and your dataset is ready to use.

# <span style='background :lightgreen' >3. Configuration of Model, Dataset, LoRA Adapters, and Training Hyperparameters</span>
- The code cell below defines the base model, dataset source, output directories, LoRA configuration, and training hyperparameters required for fine-tuning.
- It centralizes all experiment settings such as batch size, learning rate, and sequence length to make the training pipeline easy to configure and reproduce.
- Finally, it creates the required output directories for saving LoRA adapters and the merged final model.

In [ ]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

# ── Model & Data ───────────────────────────────────────────────
BASE_MODEL_ID   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"          # Base model to fine-tune (1.1B parameters)
HF_DATASET_NAME = "arif-butt/arifbutt_dataset"                 # Dataset on HuggingFace Hub for training
ADAPTER_OUTPUT_DIR      = "./tinyllama-trl-lora-adapter"         # Where to save just the LoRA adapter weights (few MB)
MERGED_OUTPUT_DIR  = "./tinyllama-trl-merged"                   # Where to save merged model (base + LoRA combined) for inference

SYSTEM_PROMPT = (
    "You are a helpful assistant."                               # System instruction prepended to every training example
)

# ── LoRA Hyperparameters ───────────────────────────────────────
LORA_R       = 16      # adapter rank (higher = more capacity, more memory)
LORA_ALPHA   = 32      # scaling factor = alpha / r = 2.0 (controls LoRA contribution strength)
LORA_DROPOUT = 0.05    # regularization dropout (prevents overfitting, 0 = no dropout)

# ── Training Hyperparameters ───────────────────────────────────
NUM_EPOCHS     = 3      # Number of full passes through the dataset
BATCH_SIZE     = 1      # Samples per GPU (small due to model size, gradient accumulation compensates)
GRAD_ACCUM     = 2      # Accumulate gradients over 2 steps (effective batch size = 1 * 2 = 2)
LEARNING_RATE  = 2e-4   # Learning rate for optimizer (higher than full fine-tuning due to LoRA)
MAX_SEQ_LENGTH = 128    # Maximum tokens per example (truncates longer sequences)
warmup_step    = 10     # Number of steps for learning rate warmup
SAVE_STEPS     = 50     # Save checkpoint every N steps
LOGGING_STEPS  = 10     # Log metrics every N steps
SEED           = 42     # Random seed for reproducibility

# ── Speed settings, picked automatically for your GPU ──────────
# Ampere or newer GPUs (RTX 30xx/40xx/50xx, A100, ...) have fast bf16 support: load the model in bf16 and use a batch
# of 8 → training is about 10x faster (RTX 5070 Ti: 2.3 min instead of 21.8 min) with answers of the same quality.
# Older GPUs such as Colab's T4 keep the original settings: fp32 weights + fp16 mixed precision, batch 1 x 2.
FAST_GPU = CAN_TRAIN and torch.cuda.get_device_capability(0)[0] >= 8
if FAST_GPU:
    BATCH_SIZE = 8      # 8 samples per step fit easily (peak ≈ 6.3 GB)
    GRAD_ACCUM = 1      # effective batch size = 8 * 1 = 8
# Gradient checkpointing (on by default in TRL) saves memory by recomputing activations, but makes training ~2.5x slower:
# only use it on GPUs with less than 12 GB of memory.
GRAD_CHECKPOINTING = CAN_TRAIN and torch.cuda.get_device_properties(0).total_memory / 1e9 < 12

print("Config ready!")
if CAN_TRAIN:
    print(f"Speed settings for {torch.cuda.get_device_name(0)}: "
          f"{'bf16 weights' if FAST_GPU else 'fp32 weights + fp16 mixed precision'}, batch {BATCH_SIZE} x {GRAD_ACCUM}, "
          f"gradient checkpointing {'on' if GRAD_CHECKPOINTING else 'off'}")

# <span style='background :lightgreen' >4. Load Base Model + Tokenizer</span>

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading TinyLlama base model...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=torch.bfloat16 if FAST_GPU else torch.float32,   # bf16 on Ampere+ GPUs (half the memory, much faster), else full fp32 precision
    device_map="auto",            # auto-place layers on GPU
    trust_remote_code=True,
)
model.config.use_cache = False    # required during training

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True,
)
tokenizer.pad_token    = tokenizer.eos_token    # no dedicated pad token
tokenizer.padding_side = "right"                # right-pad for causal LM

print("Model + Tokenizer ready!")
print(f"Vocab size  : {tokenizer.vocab_size:,}")
print(f"Model dtype : {next(model.parameters()).dtype}")

# <span style='background :lightgreen' >5. Prepare Dataset</span>

In [4]:
from datasets import load_dataset
from dotenv import load_dotenv      # Imports the load_dotenv function, which loads environment variables from a .env file into your Python environment.
from huggingface_hub import login   # Imports the login() function that allows you to authenticate with the Hugging Face Hub using an access token.
import os                           # Imports Python’s built-in os module so you can access environment variables (like your HF token).

# The following login() call is only necessary if you want to access private repos, or to use private models or to push models/datasets
'''
load_dotenv('../keys/.env', override=True)
hf_token = os.getenv('HF_TOKEN')
login(token=hf_token)
'''

raw_dataset = load_dataset("arif-butt/arifbutt_dataset")
print("✓ Dataset loaded successfully!")
raw_dataset

drarif.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1926 [00:00<?, ? examples/s]

✓ Dataset loaded successfully!


DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 1926
    })
})

In [5]:
print("\nSample examples from the training split:\n")
for idx in range(5):
    sample = raw_dataset["train"][idx]
    print(f"Example {idx + 1}: {sample}")


Sample examples from the training split:

Example 1: {'messages': [{'role': 'user', 'content': 'Who is Dr. Muhammad Arif Butt?'}, {'role': 'assistant', 'content': 'Dr. Muhammad Arif Butt is an accomplished Assistant Professor at the Department of Data Science, University of the Punjab (PU), Lahore, Pakistan. He has over 33 years of experience in teaching and management, with expertise in cybersecurity, operating systems, and artificial intelligence.'}]}
Example 2: {'messages': [{'role': 'user', 'content': 'Tell me about Dr. Arif Butt'}, {'role': 'assistant', 'content': "Dr. Muhammad Arif Butt is an Assistant Professor at Punjab University's Data Science department. He's a technology entrepreneur and cybersecurity expert with 33+ years of experience, combining academic excellence with industry innovation."}]}
Example 3: {'messages': [{'role': 'user', 'content': "What is Arif butt's current position?"}, {'role': 'assistant', 'content': 'Dr. Muhammad Arif Butt currently serves as an Assi

## Format the [arif-butt/arifbutt_dataset](https://huggingface.co/datasets/arif-butt/arifbutt_dataset)
- The dataset uses a `messages` format — a list of `{role, content}` dicts. We must convert each example into the exact chat-template string TinyLlama-Chat expects (a Zephyr-style template with `<|system|>`, `<|user|>`, `<|assistant|>` tags and `</s>` after every turn — note that this is *not* ChatML, which uses `<|im_start|>` / `<|im_end|>`):

```
<|system|>
You are a helpful assistant.</s>
<|user|>
Who is Dr. Arif Butt?</s>
<|assistant|>
Dr. Muhammad Arif Butt is an Assistant Professor...</s>
```

**Steps inside `format_example()`:**
1. Reads the `messages` list (user / assistant turns) of each example
2. Builds the TinyLlama chat-template string for each example
3. `dataset.map()` applies this to every example in the dataset

**Tokenization:**
- `labels = input_ids` — for causal LM, the model learns to predict the next token, so labels are the same as inputs
- `truncation=True` — examples longer than `MAX_SEQ_LENGTH` are cut off
- `padding="max_length"` — shorter examples are padded to the same length

In [ ]:
################# CONVERT DATASET ROWS INTO TINYLLAMA CHAT FORMAT ###############################

# ── Format Function ───────────────────────────────────────────────────────────
def format_example(example):
    """Format messages into TinyLlama chat-template conversation."""
    text = "<|system|>\nYou are a helpful assistant.</s>\n"  # start every conversation with a system prompt that sets the assistant's behaviour and persona
    for msg in example["messages"]:         # iterate over each message in the conversation — each message is a dict with "role" and "content" keys
        role    = msg["role"]               # extract the role of the message sender — either "user" or "assistant"
        content = msg["content"].strip()    # extract and clean the message text — strip() removes any leading or trailing whitespace
        if role == "user":
            text += f"<|user|>\n{content}</s>\n"        # wrap user message in TinyLlama's chat-template user tags — <|user|> opens and </s> closes the turn
        elif role == "assistant":
            text += f"<|assistant|>\n{content}</s>\n"   # wrap assistant message in TinyLlama's chat-template assistant tags — model learns to generate text between these tags
    return {"formatted_text": text}   # return the fully formatted conversation as a new column called "formatted_text" that replaces the original "messages" column

# ── Tokenize Function ─────────────────────────────────────────────────────────
def tokenize(example):
    """Convert formatted text into token IDs."""
    result = tokenizer(
                    example["formatted_text"],      # pass the chat-template formatted conversation string to the tokenizer for conversion to token IDs
                    truncation=True,                # if the formatted text exceeds MAX_SEQ_LENGTH tokens — cut it off and keep only the first MAX_SEQ_LENGTH tokens
                    max_length=MAX_SEQ_LENGTH,      # set the fixed sequence length to MAX_SEQ_LENGTH — all sequences will be either truncated or padded to exactly this length
                    padding="max_length",           # if the formatted text is shorter than MAX_SEQ_LENGTH — pad it on the right with the pad token (EOS token) to reach exactly MAX_SEQ_LENGTH tokens
                )
    result["labels"] = result["input_ids"].copy()   # for causal language model training the labels are identical to input_ids — the model learns to predict the next token by shifting input one position right internally
    return result   # return tokenized output containing input_ids, attention_mask and labels — all of length MAX_SEQ_LENGTH

# ── Apply Formatting ──────────────────────────────────────────────────────────
formatted_dataset = raw_dataset.map(
                        format_example,                                    # apply the format_example function to every row in the dataset
                        remove_columns=raw_dataset["train"].column_names,  # drop all original columns (messages, etc.) and keep only the new "formatted_text" column
                        )

# ── Tokenize Dataset ──────────────────────────────────────────────────────────
tokenized_dataset = formatted_dataset.map(
                        tokenize,                          # apply the tokenize function to every row in the formatted dataset
                        batched=True,                      # process multiple examples at once for significantly faster tokenization
                        remove_columns=["formatted_text"], # drop the "formatted_text" string column — no longer needed after tokenization
                        )

# ── Train / Eval Split ────────────────────────────────────────────────────────
spl = tokenized_dataset["train"].train_test_split(test_size=0.1, seed=SEED)  # randomly split the tokenized training data into 90% train and 10% evaluation using fixed seed for reproducibility

train_dataset = spl["train"]  # assign the 90% split to train_dataset — used to update LoRA adapter weights during training
eval_dataset  = spl["test"]   # assign the 10% split to eval_dataset  — used to monitor loss on unseen examples during training without updating any weights

print("Dataset prepared")
print(f"Train examples:\n {train_dataset}")  # print the train dataset object showing number of examples, column names and data types
print(f"Eval examples:\n {eval_dataset}")    # print the eval  dataset object showing number of examples, column names and data types

In [7]:
# Print fully tokenized training example that will be fed to the model during fine-tuning of TinyLlama-1.1B-Chat-v1.0.
print(train_dataset[0])

{'input_ids': [1, 529, 29989, 5205, 29989, 29958, 13, 3492, 526, 263, 8444, 20255, 29889, 2, 29871, 13, 29966, 29989, 1792, 29989, 29958, 13, 25125, 4942, 29889, 826, 361, 505, 263, 6189, 29257, 3236, 373, 670, 14711, 8242, 29973, 2, 29871, 13, 29966, 29989, 465, 22137, 29989, 29958, 13, 8241, 29892, 4942, 29889, 29610, 826, 361, 1205, 29873, 29915, 29879, 14711, 8242, 525, 29931, 799, 29876, 2973, 826, 361, 29915, 756, 263, 6189, 29257, 3236, 411, 29871, 29945, 19707, 3625, 363, 8041, 29889, 2, 29871, 13, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [8]:
sample = train_dataset[0]
print("Decoded text:\n")
print(tokenizer.decode(sample["input_ids"]))

Decoded text:

<s> <|system|>
You are a helpful assistant.</s> 
<|user|>
Does Dr. Arif have a Machine Learning course on his YouTube channel?</s> 
<|assistant|>
Yes, Dr. Muhammad Arif Butt's YouTube channel 'Learn With Arif' has a Machine Learning course with 5 videos available for students.</s> 
</s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s>


# <span style='background :lightgreen' >6. Configure and Inject LoRA Adapters</span>

In [9]:
lora_config = LoraConfig(
    r=LORA_R,                     # adapter rank (16)
    lora_alpha=LORA_ALPHA,        # scaling factor (32)
    lora_dropout=LORA_DROPOUT,    # dropout (0.05)
    bias="none",                  # do not adapt bias terms
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",    # attention layers
        "gate_proj", "up_proj", "down_proj",        # MLP layers
    ],
)
# WHY SOMETIMES WE DON'T USE get_peft_model IN TRL:
# - TRL's SFTTrainer has built-in LoRA support via peft_config parameter
# - When using SFTTrainer with peft_config, it automatically calls get_peft_model internally
# - This means we don't need to manually wrap the model before passing to trainer

print("LoRA config ready!")
print(f"Rank (r)        : {LORA_R}")
print(f"Alpha           : {LORA_ALPHA}")
print(f"Effective scale : {LORA_ALPHA / LORA_R}")
print(f"Target modules  : {lora_config.target_modules}")

LoRA config ready!
Rank (r)        : 16
Alpha           : 32
Effective scale : 2.0
Target modules  : {'o_proj', 'v_proj', 'up_proj', 'down_proj', 'gate_proj', 'q_proj', 'k_proj'}


# <span style='background :lightgreen' >7. Configure Training Parameters</span>

`SFTConfig` sets all training hyperparameters and passes them to `SFTTrainer`.

| Setting | Value | Explanation |
|---|---|---|
| `per_device_train_batch_size` | 1, or 8 on Ampere+ GPUs (`BATCH_SIZE`) | Samples processed per GPU per step |
| `gradient_accumulation_steps` | 2, or 1 on Ampere+ GPUs (`GRAD_ACCUM`) | Gradients accumulate over this many steps before an update → effective batch = 1 × 2 = 2 (or 8 × 1 = 8) |
| `fp16` | True on older CUDA GPUs (T4) | Mixed-precision training: the forward/backward pass runs in fp16 while the fp32 master weights are kept → faster and less activation memory |
| `bf16` | True on Ampere+ GPUs (`FAST_GPU`) | The model itself is loaded in bf16: half the memory of fp32 and much faster on RTX 30xx/40xx/50xx GPUs |
| `gradient_checkpointing` | False on GPUs with ≥ 12 GB (`GRAD_CHECKPOINTING`) | When on, activations are recomputed during backprop to save memory; this makes training ~2.5× slower, so it is used only on small GPUs (TRL turns it on by default) |
| `lr_scheduler_type` | cosine | Learning rate warms up, then decays smoothly along a cosine curve |
| `warmup_steps` | 10 (`warmup_step`) | The first 10 steps use linear warmup to avoid large early updates |
| `max_length` | 128 (`MAX_SEQ_LENGTH`) | Maximum tokens per example (this argument was called `max_seq_length` in older TRL versions) |
| `save_total_limit` | 2 | Keep only the 2 most recent checkpoints to save disk space |
| `dataset_text_field` | "text" | Only used when the dataset contains raw text. Our dataset is already tokenized (`input_ids`, `attention_mask`, `labels`), so `SFTTrainer` uses it as-is and ignores this field |

In [ ]:
training_args = SFTConfig(
    output_dir=ADAPTER_OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_steps=warmup_step,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available() and not FAST_GPU,   # fp16 mixed precision on older GPUs (needs CUDA: on CPU it would raise an error)
    bf16=FAST_GPU,                      # bf16 on Ampere or newer GPUs
    gradient_checkpointing=GRAD_CHECKPOINTING,   # off on GPUs with >= 12 GB: ~2.5x faster training
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    max_length=MAX_SEQ_LENGTH,          # renamed from `max_seq_length` in TRL >= 0.20
    dataset_text_field="text",          # ignored here: our dataset is already tokenized
    seed=SEED,
    report_to="none", # Add this line to disable wandb logging
)

print("Training config ready!")
print(f"Epochs               : {NUM_EPOCHS}")
print(f"Effective batch size : {BATCH_SIZE * GRAD_ACCUM}")
print(f"Learning rate        : {LEARNING_RATE}")
print(f"Max sequence length  : {MAX_SEQ_LENGTH} tokens")

# <span style='background :lightgreen' >8. Train the Model (needs an NVIDIA GPU)</span>
**What happens inside `trainer.train()`:**
```
For each batch:
  1. Forward pass   — feed tokens through model, get predictions
  2. Loss           — measure how wrong predictions are (cross-entropy)
  3. Backprop       — calculate gradients
  4. Update         — adjust only LoRA adapter weights
  5. Repeat
```

**How to read the training table:**
```
`Step`            | How many batches have been processed
`Training Loss`   | Loss on training data — should go down
`Validation Loss` | Loss on unseen data — should follow training loss
```
**Diagnosing your training:**
-  Both losses going down together → healthy training
-  Train loss low, eval loss rising → overfitting → reduce epochs
-  Both losses staying high → underfitting → increase epochs or LORA_R

**This cell runs only if the runtime check in section 1 found a suitable NVIDIA (CUDA) GPU** — otherwise it prints a message and is skipped.

| Hardware | Time to run this cell |
|---|---|
| Google Colab, Tesla T4 (16 GB) | ≈ 25 min (measured with gradient checkpointing on; now shorter) |
| Your PC: NVIDIA GeForce RTX 5070 Ti (16 GB) | 2.3 min (bf16, batch 8: 651 steps, peak GPU memory 6.3 GB) |

You can see the total steps, train loss, training time and peak GPU memory in the output of this cell.

In [ ]:
if CAN_TRAIN:
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,   # `tokenizer=` was renamed to `processing_class` in TRL / transformers 5
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        peft_config=lora_config,      # SFTTrainer injects LoRA automatically
        args=training_args,
    )

    import time
    print("Training started...")
    start_time = time.perf_counter()
    train_result = trainer.train()
    print("Training complete!")
    print(f"Train loss      : {train_result.training_loss:.4f}")
    print(f"Total steps     : {train_result.global_step}")
    TRAINED = True
    train_minutes = (time.perf_counter() - start_time) / 60
    print(f"Training time   : {train_minutes:.1f} min on {torch.cuda.get_device_name(0)}")
    print(f"Peak GPU memory : {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
else:
    print('⏭️  Skipping training: no suitable NVIDIA (CUDA) GPU on this machine (see the runtime check in section 1).')
    print('    To train, open this notebook in Google Colab with a T4 GPU. The remaining cells still run and use the')
    print('    fine-tuned model that is already published on the Hugging Face Hub.')

# <span style='background :lightgreen' >9. Training Loss Plot</span>
This cell plots the training and evaluation loss as a line chart so you can visually diagnose training health.

- 🔵 **Blue line** — Training loss
- 🔴 **Red dashed line** — Evaluation loss

Both lines going down smoothly and staying close together = healthy training. The plot is also saved as `training_loss.png`.

In [ ]:
if TRAINED:
    import matplotlib.pyplot as plt

    logs = trainer.state.log_history

    train_loss = [(x['step'], x['loss'])      for x in logs if 'loss' in x and 'eval_loss' not in x]
    eval_loss  = [(x['step'], x['eval_loss']) for x in logs if 'eval_loss' in x]

    plt.figure(figsize=(10, 5))
    plt.plot(*zip(*train_loss), label='Training Loss',   color='royalblue', marker='o', markersize=3)
    plt.plot(*zip(*eval_loss),  label='Validation Loss', color='tomato',    marker='s', markersize=4)
    plt.xlabel('Steps')
    plt.ylabel('Loss')
    plt.title('Training vs Validation Loss — TinyLlama Finetuning')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('loss_curve.png', dpi=150)
    plt.show()
    print("Graph saved → loss_curve.png")
else:
    print('⏭️  No training was run in this session, so there is no loss curve to plot.')

# <span style='background :lightgreen' >10. Save the Adapter Weights, Push to HF and Do Inference</span>

## a. Save the Adapter Weights
- After training we save **only the LoRA adapter weights** (48.16 MB), not the full merged model weights which will be greater than 2.2 GB.
- The saved folder contains:
    - `adapter_config.json`       — LoRA architecture definition
    - `adapter_model.safetensors` — the actual trained weights, around 48 MB only
    - `tokenizer files`           — if tokenizer saved alongside adapters
- Can be pushed on hugging face hub
- The advantages of this option are:
    -  The size of files to be saved is small
    -  If pushed on HF hub, any one having TinyLlama base can download the adapter weights and use them with base model

####  After `model.save_pretrained()` and `tokenizer.save_pretrained()`, the ADAPTER_OUTPUT_DIR (`./tinyllama-trl-lora-adapter`) contains:

| File | Size | Contents |
|---|---|---|
| `adapter_model.safetensors` | 48.16 MB | The **LoRA adapter weights only** — the small delta matrices (A & B) injected into the base model layers. Does **not** contain base model weights. |
| `adapter_config.json` | 737 B | LoRA configuration: rank (`r`), alpha, dropout, target modules, task type. Needed to reconstruct the PEFT model. |
| `tokenizer.json` | 1.76 MB | Fast tokenizer data — full vocabulary, BPE merge rules, and special token mappings. |
| `tokenizer_config.json` | 1.3 KB | Tokenizer metadata — class name, special tokens (`bos_token`, `eos_token`, `pad_token`), padding side, etc. |
| `tokenizer.model` | 488.0 KB | The raw SentencePiece model file used by the slow tokenizer for subword tokenization. |
| `special_tokens_map.json` | 437 B | Maps special token names (`bos_token`, `eos_token`, `unk_token`) to their string values. |
| `checkpoint-350/` | 4.0 KB | Mid-training checkpoint saved automatically at step 350. |
| `checkpoint-400/` | 4.0 KB | Mid-training checkpoint saved automatically at step 400 (latest). |
| `README.md` | 5.0 KB | Auto-generated model card describing the adapter. |



>- **Limitation of this option is, the uploaded adapter weights cannot be used as a standalone model, for which you have to merge**

In [ ]:
if TRAINED:
    # Save the trained LoRA adapter weights and configuration
    trainer.save_model(ADAPTER_OUTPUT_DIR)             # LoRA adapter weights + adapter config

    # Save the tokenizer files alongside the adapter for easy loading later
    tokenizer.save_pretrained(ADAPTER_OUTPUT_DIR)       # Tokenizer files alongside the adapter
    # Saves: tokenizer.json, tokenizer_config.json, special_tokens_map.json, etc.
    # Required because tokenizer must match the base model's vocabulary

    print(f"LoRA adapter saved → {ADAPTER_OUTPUT_DIR}")
else:
    print('⏭️  Skipping save: no adapter was trained in this session (the inference cells below use the adapter published on the Hub).')

In [ ]:
import os

if os.path.exists(os.path.join(ADAPTER_OUTPUT_DIR, "adapter_config.json")):
    print(f"Files saved in {ADAPTER_OUTPUT_DIR}:\n")
    for file in sorted(os.listdir(ADAPTER_OUTPUT_DIR)):
        size = os.path.getsize(os.path.join(ADAPTER_OUTPUT_DIR, file))
        print(f"  {file:40s} {size / 1e6:.2f} MB")
else:
    print('⏭️  No saved adapter found: run the training and save steps first.')

## b. Push the Files to HuggingFace Hub
Push the LoRA adapter files to the HuggingFace Hub, so anyone who has the TinyLlama base model can download and use your adapter.
- The repo is created under **your own** Hugging Face username (read from your token), e.g. `your-username/<repo-name>`.

**Before pushing, you need a Write token:**  
Go to → **huggingface.co → Profile → Settings → Access Tokens → New Token → Type: Write**

In [ ]:
import os

if os.path.exists(os.path.join(ADAPTER_OUTPUT_DIR, "adapter_model.safetensors")):
    from huggingface_hub import HfApi, create_repo, login
    from getpass import getpass

    # 1. LOGIN (SECURE INPUT) — needs a token of type "Write"
    hf_token = getpass("Enter your Hugging Face API token: ")
    login(token=hf_token)

    # 2. REPO DETAILS — the repo is created under YOUR Hugging Face username
    api = HfApi()
    repo_id = f"{api.whoami()['name']}/tinyllama-trl-lora-adapter"

    # 3. CREATE REPO (IF NOT EXISTS)
    print("Creating repo (if needed)...")
    create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    print(f"Repo ready: https://huggingface.co/{repo_id}")

    # 4. UPLOAD FILES
    print("Uploading adapter files (this may take a while)...")
    api.upload_folder(folder_path=ADAPTER_OUTPUT_DIR, repo_id=repo_id, repo_type="model")

    print("Upload complete!")
    print(f"\nYour adapter files is live at: https://huggingface.co/{repo_id}")
else:
    print('⏭️  Skipping upload: no local adapter files found (they are created by the cells above, after training).')

## c. Inference with loading LoRa Adapter from hugging face

In [ ]:
# This script loads a base model with pre-trained LoRA adapters directly from HuggingFace Hub
# and sets up a text generation pipeline for inference without needing to merge adapters locally

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftModel

# ── Model Identifiers ────────────────────────────────────────────────────────────────────────
# 🔹 Hugging Face repo names — these point to models stored on the HuggingFace Hub
BASE_MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"      # base pre-trained model — 1.1B parameter chat-optimized Llama variant
ADAPTER_REPO = "arif-butt/tinyllama-trl-lora-adapter"    # LoRA adapter weights uploaded to HuggingFace Hub — trained on custom dataset

# ── Load Base Model ──────────────────────────────────────────────────────────────────────────
# First load the original pre-trained model without any adapters
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,                                        # identifier for base model on HuggingFace Hub
    dtype=torch.float16,                            # use float16 precision — reduces memory usage by 50% with minimal quality loss
    device_map="auto"                                     # automatically distribute layers across available GPUs/CPU
)

# Load the trained LoRA adapter weights and attach them to the base model
# The adapter_config.json in the repo defines which layers to modify and with what rank
model = PeftModel.from_pretrained(
    base_model,                                           # base model to attach adapters to — original weights remain frozen
    ADAPTER_REPO                                          # HuggingFace repo containing adapter weights and config
)

print("Base model loaded and LoRA adapters attached successfully")

# Tokenizer must match the one used during training — loaded from adapter repo to ensure consistency
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_REPO                                          # tokenizer config saved in same repo as adapters
)

print("Tokenizer loaded successfully")

# Pipeline simplifies inference by handling tokenization, model generation, and decoding automatically
# This creates a ready-to-use interface for generating text with the fine-tuned model
pipe = pipeline(
    "text-generation",                                    # task type — generates text conditioned on input prompt
    model=model,                                          # fine-tuned model with LoRA adapters attached
    tokenizer=tokenizer                                   # tokenizer that matches the model's vocabulary
)

print("Text generation pipeline created")
print(f"Model loaded from: {BASE_MODEL_ID} + {ADAPTER_REPO}")
print(f"Model device: {model.device}")
print(f"Total parameters: {model.num_parameters():,}")


In [ ]:
# Test prompt
prompt = "Q: Name all the courses Arif butt teach?\nA:"
output = pipe(prompt, max_new_tokens=100, do_sample=True, temperature=0.2)
print(output[0]["generated_text"])

### Before merging, we free up VRAM by deleting the trainer and model from memory. This prevents Out-of-Memory errors when reloading the base model for merging.

In [ ]:
import gc
import torch

for name in ("trainer", "model"):      # these only exist if the earlier cells created them (e.g. training was run)
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("GPU memory cleared!")
if torch.cuda.is_available():
    print(f"Allocated : {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"Reserved  : {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# <span style='background :lightgreen' >11. Merge Adapter Weights, Push to HF and Do Inference</span>

## a. Save the Complete Merged  Model
- Now that we have saved the adapter weights, we can now create the final standalone model by merging the LoRA adapters into the base model.
- Before merge → need both base model + adapter files to run
- After merge → one single file, no PEFT library needed at inference time
- **What Gets Saved After Both Steps:**
```
./tinyllama-trl-lora-adapter/                ← ADAPTER_OUTPUT_DIR
  ├── adapter_config.json               ← LoRA config (r=16, alpha=32, target modules)
  ├── adapter_model.safetensors         ← LoRA adapter weights only (~48 MB)
  ├── tokenizer.json                    ← full tokenizer
  ├── tokenizer_config.json             ← tokenizer settings
  ├── special_tokens_map.json           ← special tokens
  └── tokenizer.model                   ← tokenizer binary

./tinyllama-trl-merged/                      ← MERGED_OUTPUT_DIR
  ├── config.json                       ← full model architecture config
  ├── generation_config.json            ← default generation settings
  ├── model.safetensors                 ← FULL merged weights (~2.2 GB)
  ├── tokenizer.json                    ← full tokenizer
  ├── tokenizer_config.json             ← tokenizer settings
  ├── special_tokens_map.json           ← special tokens
  └── tokenizer.model                   ← tokenizer binary
```

In [ ]:
import os
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Use the adapter you trained above; if you skipped training (no GPU), use the adapter published on the Hub instead
ADAPTER_SOURCE = ADAPTER_OUTPUT_DIR if os.path.exists(os.path.join(ADAPTER_OUTPUT_DIR, "adapter_config.json")) else "arif-butt/tinyllama-trl-lora-adapter"
print(f"Adapter source: {ADAPTER_SOURCE}")

# We need to reload the base model, because we deleted it in above code cell to free GPU memory. Now we reload it clean — no adapters attached yet.
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
)

# Putting a trained overlay on top of the original model. It reads the adapter architecture from `adapter_config.json`, loads the trained weights, and clips them onto the exact layers they were trained on.
print("Attaching saved LoRA adapters...")
finetuned_model = PeftModel.from_pretrained(base_model, ADAPTER_SOURCE)

#  During training, adapters lived separately alongside the frozen weights. Merging permanently bakes them in
print("Merging adapter weights into base model...")
finetuned_model = finetuned_model.merge_and_unload()     # Final Weight = Original Frozen Weight + (LoRA_B × LoRA_A) × scaling
finetuned_model.eval()      # Switches the model from training mode to inference mode by disabling dropout (randomly switching off neurons during training to prevent overfitting). During inference you want every neuron active for consistent answers.

print("Saving merged model...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_SOURCE)   # tokenizer saved together with the adapter
finetuned_model.save_pretrained(MERGED_OUTPUT_DIR)
tokenizer.save_pretrained(MERGED_OUTPUT_DIR)

print(f"Merged model saved to: {MERGED_OUTPUT_DIR}")
print(f"Files: {sorted(os.listdir(MERGED_OUTPUT_DIR))}")

## b. Push the Merged model Files to HuggingFace Hub

After merging, push the standalone model to HuggingFace Hub so it can be loaded from anywhere without needing local files.

**Before pushing, you need a Write token:**  
Go to → **huggingface.co → Profile → Settings → Access Tokens → New Token → Type: Write**
- The repo is created under **your own** Hugging Face username (read from your token).

In [ ]:
import os

if os.path.exists(os.path.join(MERGED_OUTPUT_DIR, "model.safetensors")):
    from huggingface_hub import HfApi, create_repo, login
    from getpass import getpass

    # 1. LOGIN (SECURE INPUT) — needs a token of type "Write"
    hf_token = getpass("Enter your Hugging Face API token: ")
    login(token=hf_token)

    # 2. REPO DETAILS — the repo is created under YOUR Hugging Face username
    api = HfApi()
    repo_id = f"{api.whoami()['name']}/tinyllama-trl-merged"

    # 3. CREATE REPO (IF NOT EXISTS)
    print("Creating repo (if needed)...")
    create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    print(f"Repo ready: https://huggingface.co/{repo_id}")

    # 4. UPLOAD FILES
    print("Uploading merged model (this may take a while)...")
    api.upload_folder(folder_path=MERGED_OUTPUT_DIR, repo_id=repo_id, repo_type="model")

    print("Upload complete!")
    print(f"\nYour merged model is live at: https://huggingface.co/{repo_id}")
else:
    print('⏭️  Skipping upload: no local merged model found (they are created by the cells above, after training).')

## c. Inference by loading merged model from hugging face

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# ──────────────────────────────
# 1. Model Repo
# ──────────────────────────────
MERGED_MODEL_ID = "arif-butt/tinyllama-trl-merged"

# ──────────────────────────────
# 2. Load tokenizer
# ──────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_ID)

# ──────────────────────────────
# 3. Load merged model (standalone)
# ──────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_ID,
    dtype=torch.float16,  # use half precision for GPU efficiency
    device_map="auto"           # automatically use GPU if available
)

# ──────────────────────────────
# 4. Create pipeline
# ──────────────────────────────
pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False  # only return the model-generated answer
)



config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.jinja:   0%|          | 0.00/410 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

In [ ]:
# ──────────────────────────────
# 5. Example prompt
# ──────────────────────────────
prompt = "Q: Who is Dr. Arif Butt?\nA:"

output = pipe(
    prompt,
    max_new_tokens=150,
    do_sample=False,           # deterministic/factual output
    repetition_penalty=1.1
)
# ──────────────────────────────
# 6. Print answer
# ──────────────────────────────
print(output[0]["generated_text"].strip())

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Dr. Muhammad Arif Butt is an Assistant Professor at the Department of Data Science, University of the Punjab (PU), Lahore, Pakistan.


# <span style='background :lightgreen' >12. Generate GGUF, Push it to HuggingFace Hub and do local Inference via Ollama and llama.cpp</span>
## a. Generate GGUF

- **GPT-Generated Unified Format (GGUF)** is a modern, unified model file format designed for efficient local inference, especially for *LLaMA-family* of models using tools like **llama.cpp** and **Ollama**.
    - *GGUF File Container:* The outer wrapper that holds all model components in a single binary file, eliminating the need for multi-file setups and ensuring consistency between model components.
    - *Metadata Layer:* This layer stores model configuration and descriptive information, including model architecture and dimensions, context length, special tokens (BOS, EOS, PAD) and inference-related parameters.
    - *Tokenization Layer:* This layer defines how raw text is converted into tokens.
    - *Quantization Layer:* This layer describes how weights are compressed using different quantization formats like Q4, and Q8. This enables running large models on limited hardware.
    - *Model Weights and Tensors:* This layer contains the actual neural network parameters stored as optimized tensors.

**How the conversion works (next two cells):**
- `git clone --depth=1 https://github.com/ggml-org/llama.cpp` downloads llama.cpp, which contains the official converter `convert_hf_to_gguf.py` and its `gguf-py` package (no compilation needed — the converter is plain Python and uses the libraries already in our environment).
- `convert_hf_to_gguf.py` converts the merged Hugging Face model in `MERGED_OUTPUT_DIR` into a single GGUF file (`TinyLlama-Trl_q8.gguf`), quantized to `q8_0`.
- `upload_folder()` from `huggingface_hub` then uploads the GGUF file to your own Hugging Face repo so it can be downloaded and run locally.

**Why `q8_0`?**
- 8-bit quantization — almost zero accuracy loss compared to fp16
- TinyLlama Q8_0 ≈ 1.1 GB — runs easily on any laptop CPU
- Best balance of size and accuracy for a domain-specific Q&A model

| GGUF Format | Size | Accuracy |
|---|---|---|
| `q8_0` | ~1.1 GB | Almost no loss |
| `q5_k_m` | ~0.7 GB | Very small loss |
| `q4_k_m` | ~0.6 GB | Slight loss |

> `convert_hf_to_gguf.py` can write `f32`, `f16`, `bf16` and `q8_0` directly. The `q5_k_m` / `q4_k_m` formats need the compiled `llama-quantize` tool from llama.cpp.

In [ ]:
# Download llama.cpp (only its Python converter is used, nothing is compiled)
import os, subprocess

LLAMA_CPP_DIR = "./llama.cpp"
if not os.path.exists(os.path.join(LLAMA_CPP_DIR, "convert_hf_to_gguf.py")):
    subprocess.run(["git", "clone", "--depth=1", "https://github.com/ggml-org/llama.cpp", LLAMA_CPP_DIR], check=True)
print("llama.cpp converter ready:", os.path.join(LLAMA_CPP_DIR, "convert_hf_to_gguf.py"))

In [ ]:
import os, sys, subprocess

GGUF_DIR  = "./tinyllama-trl-gguf"                                     # output folder for the GGUF file
GGUF_PATH = os.path.join(GGUF_DIR, "TinyLlama-Trl_q8.gguf")
os.makedirs(GGUF_DIR, exist_ok=True)

if os.path.exists(os.path.join(MERGED_OUTPUT_DIR, "config.json")):
    print(f"Converting {MERGED_OUTPUT_DIR} -> {GGUF_PATH} (q8_0) ...")
    subprocess.run([
        sys.executable, os.path.join(LLAMA_CPP_DIR, "convert_hf_to_gguf.py"),
        MERGED_OUTPUT_DIR,                # merged Hugging Face model (config.json + model.safetensors + tokenizer)
        "--outfile", GGUF_PATH,           # single output GGUF file
        "--outtype", "q8_0",              # 8-bit quantization
    ], check=True)
    print(f"GGUF conversion done! {GGUF_PATH} ({os.path.getsize(GGUF_PATH) / 1e9:.2f} GB)")
else:
    print('⏭️  Skipping conversion: no merged model found in MERGED_OUTPUT_DIR (run the merge cell above first).')

## b. Push GGUF file on Hugging Face Hub

In [ ]:
import os

if os.path.exists(os.path.join(GGUF_DIR, "TinyLlama-Trl_q8.gguf")):
    from huggingface_hub import HfApi, create_repo, login
    from getpass import getpass

    # 1. LOGIN (SECURE INPUT) — needs a token of type "Write"
    hf_token = getpass("Enter your Hugging Face API token: ")
    login(token=hf_token)

    # 2. REPO DETAILS — the repo is created under YOUR Hugging Face username
    api = HfApi()
    repo_id = f"{api.whoami()['name']}/tinyllama-trl-gguf"

    # 3. CREATE REPO (IF NOT EXISTS)
    print("Creating repo (if needed)...")
    create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    print(f"Repo ready: https://huggingface.co/{repo_id}")

    # 4. UPLOAD FILES
    print("Uploading GGUF file (this may take a while)...")
    api.upload_folder(folder_path=GGUF_DIR, repo_id=repo_id, repo_type="model")

    print("Upload complete!")
    print(f"\nYour GGUF file is live at: https://huggingface.co/{repo_id}")
else:
    print('⏭️  Skipping upload: no local GGUF file found (they are created by the cells above, after training).')

## c. Inference by using GGUF from Hugging Face Hub using Ollama or llama.cpp
<h1 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Ollama lets you download models from Ollama Hub or the Hugging Face Hub.</div></h1>

### Step 1: Download and Install Ollama
- Download Ollama for your Mac, Linux or Windows machine by visiting: https://ollama.com/download
- To check if ollama has been installed on your machine, open a terminal and give the command `ollama`:
```
Available Commands:
  serve       Start ollama
  create      Create a model
  show        Show information for a model
  run         Run a model
  stop        Stop a running model
  pull        Pull a model from a registry
  push        Push a model to a registry
  signin      Sign in to ollama.com
  signout     Sign out from ollama.com
  list        List models
  ps          List running models
  cp          Copy a model
  rm          Remove a model
  launch      Launch an integration with Ollama
  help        Help about any command
Use "ollama [command] --help" for more information about a specific command.
```
- To ensure Ollama application is running in the background, just open a browser and go to http://localhost:11434/ and it will display a message saying **"ollama is running"**. If this is not the case, then on the terminal give the command `ollama serve` and try again :)

### Step 2: Download the GGUF version of your Fine-Tuned model
- Ollama only runs models in the GGUF format because it is built on llama.cpp which reads GGUF files for efficient inference
- Go to **https://huggingface.co/arif-butt/tinyllama-trl-gguf**, click ‘Files and Versions’ on the model page, and then click the Download button on **https://huggingface.co/arif-butt/tinyllama-trl-gguf/blob/main/TinyLlama-Trl_q8.gguf** to download the model on your local machine
- If you did not convert your own GGUF above, the cell after the next one downloads this published file for you (into `./tinyllama-trl-gguf/`).

### Step 3: Create a Modelfile
- A **Modelfile** in Ollama is a small text configuration file that tells Ollama how to use a specific model.
- Think of it as a bridge between the GGUF model file and Ollama’s chat interface, it defines which model to load and how the conversation should behave.
- **Why is it needed?** Ollama itself cannot automatically know:
  - Which GGUF model to load.
  - How to handle prompts, messages, and system instructions.
  - What generation parameters to use (temperature, top-p, etc.).
- The Modelfile encapsulates all this information so you can run your model with consistent behavior.
- It makes customizing models easier without modifying the GGUF file itself.
- The next cell writes the `Modelfile` for you, right next to the GGUF file (it uses the GGUF you converted above, or downloads the published one from the Hub if you have none). Its content looks like this:
```
FROM <path-to-your>/TinyLlama-Trl_q8.gguf
SYSTEM """You are a helpful assistant."""

TEMPLATE """<|system|>
{{ .System }}</s>
<|user|>
{{ .Prompt }}</s>
<|assistant|>
"""

PARAMETER stop "<|system|>"
PARAMETER stop "<|user|>"
PARAMETER stop "<|assistant|>"
PARAMETER stop "</s>"
PARAMETER temperature 0.7
PARAMETER num_ctx 256
```

In [ ]:
import os, shutil, subprocess
from huggingface_hub import hf_hub_download

GGUF_DIR  = globals().get("GGUF_DIR", "./tinyllama-trl-gguf")
GGUF_PATH = os.path.join(GGUF_DIR, "TinyLlama-Trl_q8.gguf")
if not os.path.exists(GGUF_PATH):                     # no local GGUF: download the published one
    print("No local GGUF found, downloading arif-butt/tinyllama-trl-gguf from the Hugging Face Hub ...")
    GGUF_PATH = hf_hub_download("arif-butt/tinyllama-trl-gguf", "TinyLlama-Trl_q8.gguf", local_dir=GGUF_DIR)

MODELFILE_PATH = os.path.join(GGUF_DIR, "Modelfile")
modelfile = '''FROM {gguf_path}
SYSTEM """You are a helpful assistant."""

TEMPLATE """<|system|>
{{ .System }}</s>
<|user|>
{{ .Prompt }}</s>
<|assistant|>
"""

PARAMETER stop "<|system|>"
PARAMETER stop "<|user|>"
PARAMETER stop "<|assistant|>"
PARAMETER stop "</s>"
PARAMETER temperature 0.7
PARAMETER num_ctx 256
'''.replace("{gguf_path}", os.path.abspath(GGUF_PATH))

with open(MODELFILE_PATH, "w", encoding="utf-8") as f:
    f.write(modelfile)
print(f"Modelfile written to {MODELFILE_PATH}:\n")
print(modelfile)

def ollama(*args):
    # Run an `ollama` CLI command if Ollama is installed, otherwise explain how to get it
    if shutil.which("ollama") is None:
        print("⏭️  Ollama is not installed (or not on PATH): install it from https://ollama.com/download and run this cell again.")
        return
    result = subprocess.run(["ollama", *args], capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(result.stdout.strip() or result.stderr.strip())

In [ ]:
ollama("create", "tinyllama-ft-trl", "-f", MODELFILE_PATH)

In [ ]:
ollama("list")

### Step 5 - Run your new Model via CLI

In [ ]:
ollama("run", "tinyllama-ft-trl", "What is the email address of Dr. Arif Butt?")

## Accessing via llama.cpp
Install llama.cpp from https://github.com/ggml-org/llama.cpp/releases (or `brew install llama.cpp` on Mac), then:
#### Via Command Line Terminal:
```bash
$ llama-cli -m ./tinyllama-trl-gguf/TinyLlama-Trl_q8.gguf
```
<div style="text-align:center;">
    <img src="../images/llamacpp-ft.png"
         style="max-width:1000px; width:100%; height:auto; display:inline-block;">
</div>

#### Via Browser:
- You can start the llama-server and load the model. The embedded server often exposes a lightweight Web UI at `http://localhost:PORT` where you can interact with the model via browser.
```bash
$ llama-server -m ./tinyllama-trl-gguf/TinyLlama-Trl_q8.gguf   --port 8080
```
<div style="text-align:center;">
    <img src="../images/llamacpp-ft2.png"
         style="max-width:1000px; width:100%; height:auto; display:inline-block;">
</div>